# Milestone 5 Notebook (Variant B) - DeBERTa / RoBERTa Ensemble

## Setup

Loads the two fine-tuned checkpoints and the test data. Replace the checkpoint
paths below with your own fine-tuned model folders (or hub repo names) from
your earlier training run.

In [1]:
#!unzip smart-mcq-solver-challenge.zip

Archive:  smart-mcq-solver-challenge.zip
  inflating: sample_submission.csv   
  inflating: test.csv                
  inflating: train.csv               


In [2]:
!pip install transformers torch -q


In [3]:
import pandas as pd
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification

compute_device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"running on {compute_device}")

MODEL_PATH_DEBERTA = "microsoft/deberta-v3-small"
MODEL_PATH_ROBERTA = "roberta-base"

tokenizer_deberta = AutoTokenizer.from_pretrained(MODEL_PATH_DEBERTA)
classifier_deberta = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH_DEBERTA, num_labels=5)
classifier_deberta.to(compute_device)
classifier_deberta.eval()

tokenizer_roberta = AutoTokenizer.from_pretrained(MODEL_PATH_ROBERTA)
classifier_roberta = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH_ROBERTA, num_labels=5)
classifier_roberta.to(compute_device)
classifier_roberta.eval()

LABEL_MAP = {0: "A", 1: "B", 2: "C", 3: "D", 4: "E"}
OPTION_LETTERS = ["A", "B", "C", "D", "E"]

test_data = pd.read_csv("test.csv")
train_data = pd.read_csv("train.csv")
print("setup done")


running on cuda


config.json:   0%|          | 0.00/578 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model: reconstructing file:   0%|          |  0.00B / 2.46MB            

spm.model: downloading bytes:           |  0.00B            

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  499MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


setup done


### Question format used for inference

The fine-tuned checkpoints expect a single input per question (not 5 separate
option inputs like the multiple-choice model in milestone 4). This builds one
string containing the prompt and all five options. Use whatever format your
checkpoints were actually fine-tuned on if it differs from this.

In [4]:
def assemble_question_string(record):
    pieces = [str(record["prompt"])]
    for opt in OPTION_LETTERS:
        pieces.append(opt + ") " + str(record[opt]))
    return " ".join(pieces)

def run_inference(net, tok, question_string):
    tokenized = tok(
        question_string,
        truncation=True,
        padding="max_length",
        max_length=256,
        return_tensors="pt",
    )
    tokenized = {key: tensor.to(compute_device) for key, tensor in tokenized.items()}
    with torch.no_grad():
        outputs = net(**tokenized)
    probabilities = F.softmax(outputs.logits, dim=1)
    return probabilities[0].detach().cpu()


## Q1 - DeBERTa prediction for row index 25

In [5]:
record_25 = test_data.iloc[25]
question_25 = assemble_question_string(record_25)

deberta_probabilities_25 = run_inference(classifier_deberta, tokenizer_deberta, question_25)

winning_index = torch.argmax(deberta_probabilities_25).item()
winning_letter = LABEL_MAP[winning_index]
winning_probability = deberta_probabilities_25[winning_index].item()

print("DeBERTa probs:", deberta_probabilities_25)
print(winning_letter + ", probability of " + winning_letter + " = " + str(round(winning_probability, 4)))


DeBERTa probs: tensor([0.2037, 0.2462, 0.2354, 0.1494, 0.1653], dtype=torch.float16)
B, probability of B = 0.2462


## Q2 - simple average of DeBERTa and RoBERTa probabilities

In [6]:
roberta_probabilities_25 = run_inference(classifier_roberta, tokenizer_roberta, question_25)

mean_probabilities_25 = (deberta_probabilities_25 + roberta_probabilities_25) / 2

mean_winning_index = torch.argmax(mean_probabilities_25).item()
mean_winning_letter = LABEL_MAP[mean_winning_index]

print("simple avg probs:", mean_probabilities_25)
print("top option:", mean_winning_letter, "prob:", round(mean_probabilities_25[mean_winning_index].item(), 4))


simple avg probs: tensor([0.2156, 0.2198, 0.2196, 0.1580, 0.1870])
top option: B prob: 0.2198


## Q3 - weighted ensemble, DeBERTa 0.70 / RoBERTa 0.30

In [7]:
DEBERTA_SHARE = 0.70
ROBERTA_SHARE = 0.30

combined_probabilities_25 = DEBERTA_SHARE * deberta_probabilities_25 + ROBERTA_SHARE * roberta_probabilities_25

combined_winning_index = torch.argmax(combined_probabilities_25).item()
combined_winning_letter = LABEL_MAP[combined_winning_index]

print("weighted probs:", combined_probabilities_25)
print("top option after weighted ensemble:", combined_winning_letter)


weighted probs: tensor([0.2108, 0.2304, 0.2259, 0.1546, 0.1783])
top option after weighted ensemble: B


## Q4 - Top-3 prediction string for row 25

In [8]:
sorted_indices_25 = torch.argsort(combined_probabilities_25, descending=True).tolist()
top_three_letters_25 = [LABEL_MAP[i] for i in sorted_indices_25[:3]]
top_three_string_25 = " ".join(top_three_letters_25)

print("top 3 for row 25:", top_three_string_25)


top 3 for row 25: B C A


## Q5 - run the weighted ensemble on all of test.csv

In [9]:
def compute_top3_for_record(record):
    question_string = assemble_question_string(record)
    p_deberta = run_inference(classifier_deberta, tokenizer_deberta, question_string)
    p_roberta = run_inference(classifier_roberta, tokenizer_roberta, question_string)
    p_combined = DEBERTA_SHARE * p_deberta + ROBERTA_SHARE * p_roberta
    sorted_idx = torch.argsort(p_combined, descending=True).tolist()
    top_three = [LABEL_MAP[i] for i in sorted_idx[:3]]
    return " ".join(top_three), p_combined

output_rows = []
for _, record in test_data.iterrows():
    prediction_string, _ = compute_top3_for_record(record)
    output_rows.append({"id": record["id"], "prediction": prediction_string})

submission_table = pd.DataFrame(output_rows)
submission_table.to_csv("submission.csv", index=False)

verification_table = pd.read_csv("submission.csv")
print("rows in submission.csv:", len(verification_table))


rows in submission.csv: 500


## Q6 - Test-Time Augmentation on the first 50 rows

Compares a single DeBERTa pass on the original prompt against the averaged
prediction from (original prompt + instruction-augmented prompt).

In [10]:
INSTRUCTION_TEXT = "Answer the following multiple-choice question carefully: "

def assemble_augmented_question_string(record):
    pieces = [INSTRUCTION_TEXT + str(record["prompt"])]
    for opt in OPTION_LETTERS:
        pieces.append(opt + ") " + str(record[opt]))
    return " ".join(pieces)

flip_counter = 0
sample_block = test_data.iloc[0:50]

for _, record in sample_block.iterrows():
    original_question = assemble_question_string(record)
    augmented_question = assemble_augmented_question_string(record)

    original_probabilities = run_inference(classifier_deberta, tokenizer_deberta, original_question)
    augmented_probabilities = run_inference(classifier_deberta, tokenizer_deberta, augmented_question)

    original_winner = torch.argmax(original_probabilities).item()

    averaged_probabilities = (original_probabilities + augmented_probabilities) / 2
    averaged_winner = torch.argmax(averaged_probabilities).item()

    if averaged_winner != original_winner:
        flip_counter += 1

print("rows with a different Top-1 after TTA:", flip_counter)


rows with a different Top-1 after TTA: 0


## Q7 - DeBERTa alone vs Weighted Ensemble, first 100 rows

In [11]:
sample_block_100 = test_data.iloc[0:100]

deberta_top1_records = []
ensemble_top1_records = []
deberta_confidence_records = []
ensemble_confidence_records = []
deberta_top3_records = []
ensemble_top3_records = []

for _, record in sample_block_100.iterrows():
    question_string = assemble_question_string(record)
    p_deberta = run_inference(classifier_deberta, tokenizer_deberta, question_string)
    p_roberta = run_inference(classifier_roberta, tokenizer_roberta, question_string)
    p_combined = DEBERTA_SHARE * p_deberta + ROBERTA_SHARE * p_roberta

    deberta_winner = torch.argmax(p_deberta).item()
    combined_winner = torch.argmax(p_combined).item()

    deberta_top1_records.append(LABEL_MAP[deberta_winner])
    ensemble_top1_records.append(LABEL_MAP[combined_winner])

    deberta_confidence_records.append(p_deberta[deberta_winner].item())
    ensemble_confidence_records.append(p_combined[combined_winner].item())

    deberta_rank = torch.argsort(p_deberta, descending=True).tolist()[:3]
    combined_rank = torch.argsort(p_combined, descending=True).tolist()[:3]
    deberta_top3_records.append([LABEL_MAP[i] for i in deberta_rank])
    ensemble_top3_records.append([LABEL_MAP[i] for i in combined_rank])

mismatch_total = sum(
    1 for a, b in zip(deberta_top1_records, ensemble_top1_records) if a != b
)
print("rows with different Top-1 (DeBERTa vs ensemble):", mismatch_total)


rows with different Top-1 (DeBERTa vs ensemble): 0


## Q8 - confidence gain from ensembling

In [12]:
confidence_deltas = [
    b - a for a, b in zip(deberta_confidence_records, ensemble_confidence_records)
]

improved_total = sum(1 for delta in confidence_deltas if delta > 0)
print("rows with positive confidence gain:", improved_total)


rows with positive confidence gain: 0


## Q9 - Top-3 ranking changes between DeBERTa alone and the ensemble

In [13]:
ranking_change_total = 0
for da, db in zip(deberta_top3_records, ensemble_top3_records):
    if da != db:
        ranking_change_total += 1

print("rows with at least one Top-3 ranking change:", ranking_change_total)


rows with at least one Top-3 ranking change: 0


## Q10 - MAP@3 on the first 100 labeled rows

test.csv has no ground truth, so this uses the first 100 rows of train.csv
(which does have an answer column) as the labeled validation slice.

In [14]:
def average_precision_top3(correct_letter, ranked_letters):
    if correct_letter in ranked_letters:
        position = ranked_letters.index(correct_letter) + 1
        return 1.0 / position
    return 0.0

validation_block = train_data.iloc[0:100]

precision_scores = []
for _, record in validation_block.iterrows():
    question_string = assemble_question_string(record)
    p_deberta = run_inference(classifier_deberta, tokenizer_deberta, question_string)
    p_roberta = run_inference(classifier_roberta, tokenizer_roberta, question_string)
    p_combined = DEBERTA_SHARE * p_deberta + ROBERTA_SHARE * p_roberta

    rank = torch.argsort(p_combined, descending=True).tolist()[:3]
    ranked_letters = [LABEL_MAP[i] for i in rank]

    precision_scores.append(average_precision_top3(record["answer"], ranked_letters))

mean_average_precision = sum(precision_scores) / len(precision_scores)
print("MAP@3:", round(mean_average_precision, 4))


MAP@3: 0.4717
